Health Canada Drug Product Database (DPD): flatten the four extract ZIPs
into products (one per DIN) and medicines (brand name + active substances).
Saves both to data/2_translated/ (dpd_products.csv, dpd_medicines.csv), linked by medicine_id;
the team has not yet decided which level becomes the entity.

Source: https://www.canada.ca/en/health-canada/services/drugs-health-products/
        drug-products/drug-product-database/read-file-drug-product-database-data-extract.html

## Setup: paths and column definitions

In [9]:
import zipfile
from pathlib import Path

import pandas as pd


# Notebooks have no __file__: walk up from the working directory to the repo root
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "1_raw").is_dir())
dpd_dir = repo_root / "data" / "1_raw"/"03. DPD"

translated_data_dir = repo_root / "data" / "2_translated"
translated_data_dir.mkdir(parents=True, exist_ok=True)

# One ZIP per product status group; file names inside carry the suffix.
extract_sets = {
    "marketed": ("allfiles.zip", ""),
    "approved": ("allfiles_ap.zip", "_ap"),
    "dormant": ("allfiles_dr.zip", "_dr"),
    "cancelled": ("allfiles_ia.zip", "_ia"),
}

# Only the 9 tables that feed the product table are loaded. Skipped:
#   package      - UPC, package size/type: packaging detail, not in the other sources
#   pharm        - pharmacopoeia standard (USP, BP, ...): sparse, not used for matching
#   vet          - veterinary species: only relevant for veterinary products
#   inactive.txt - cancelled ZIP only; repeats DIN, brand name and date from drug_ia.txt
# The files have no header row; column names come from the DPD read-me page.
# (ther.txt only ships the ATC columns, not the AHFS ones listed on the page.)
columns = {
    "drug": ["DRUG_CODE", "PRODUCT_CATEGORIZATION", "CLASS", "DRUG_IDENTIFICATION_NUMBER",
             "BRAND_NAME", "DESCRIPTOR", "PEDIATRIC_FLAG", "ACCESSION_NUMBER", "NUMBER_OF_AIS",
             "LAST_UPDATE_DATE", "AI_GROUP_NO", "CLASS_F", "BRAND_NAME_F", "DESCRIPTOR_F"],
    "comp": ["DRUG_CODE", "MFR_CODE", "COMPANY_CODE", "COMPANY_NAME", "COMPANY_TYPE",
             "ADDRESS_MAILING_FLAG", "ADDRESS_BILLING_FLAG", "ADDRESS_NOTIFICATION_FLAG",
             "ADDRESS_OTHER", "SUITE_NUMBER", "STREET_NAME", "CITY_NAME", "PROVINCE",
             "COUNTRY", "POSTAL_CODE", "POST_OFFICE_BOX", "PROVINCE_F", "COUNTRY_F"],
    "ingred": ["DRUG_CODE", "ACTIVE_INGREDIENT_CODE", "INGREDIENT", "INGREDIENT_SUPPLIED_IND",
               "STRENGTH", "STRENGTH_UNIT", "STRENGTH_TYPE", "DOSAGE_VALUE", "BASE",
               "DOSAGE_UNIT", "NOTES", "INGREDIENT_F", "STRENGTH_UNIT_F", "STRENGTH_TYPE_F",
               "DOSAGE_UNIT_F"],
    "form": ["DRUG_CODE", "PHARM_FORM_CODE", "PHARMACEUTICAL_FORM", "PHARMACEUTICAL_FORM_F"],
    "route": ["DRUG_CODE", "ROUTE_OF_ADMINISTRATION_CODE", "ROUTE_OF_ADMINISTRATION",
              "ROUTE_OF_ADMINISTRATION_F"],
    "schedule": ["DRUG_CODE", "SCHEDULE", "SCHEDULE_F"],
    "status": ["DRUG_CODE", "CURRENT_STATUS_FLAG", "STATUS", "HISTORY_DATE", "STATUS_F",
               "LOT_NUMBER", "EXPIRATION_DATE"],
    "ther": ["DRUG_CODE", "TC_ATC_NUMBER", "TC_ATC", "TC_ATC_F"],
    "biosimilar": ["DRUG_CODE", "SI_DESC_E", "SI_DESC_F", "SI_CODE"],
}

## Load the 9 tables from all four ZIPs

In [10]:
def read_table(name):
    """Read one table from all four ZIPs and stack them."""
    frames = []
    for set_name, (zip_name, suffix) in extract_sets.items():
        with zipfile.ZipFile(dpd_dir / zip_name) as zipped_file:
            with zipped_file.open(f"{name}{suffix}.txt") as table_file:
                try:
                    df = pd.read_csv(table_file, header=None, names=columns[name],
                                     dtype="string", keep_default_na=False, encoding="utf-8")
                except pd.errors.EmptyDataError:  # e.g. biosimilar_dr.txt is empty
                    df = pd.DataFrame(columns=columns[name], dtype="string")
        df = df.apply(lambda col: col.str.strip()).replace("", pd.NA)
        df["extract_set"] = set_name
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def join_unique(values):
    values = values.dropna().drop_duplicates()
    return "; ".join(values) if len(values) else pd.NA


tables = {name: read_table(name) for name in columns}
drug = tables["drug"]

## Build the product-level table (one row per DIN)

In [11]:
# Active ingredients as "NAME (strength unit)"
ingred = tables["ingred"]
strength = (ingred["STRENGTH"].fillna("") + " " + ingred["STRENGTH_UNIT"].fillna("")).str.strip()
ingred["ingredient_text"] = ingred["INGREDIENT"].str.upper() + (" (" + strength + ")").where(strength.ne(""), "")

# Manufacturer = the DIN owner (one per product)
comp = tables["comp"]
manufacturer = comp[comp["COMPANY_TYPE"].eq("DIN_OWNER")].groupby("DRUG_CODE")["COMPANY_NAME"].agg(join_unique)

status = tables["status"]
current_status = (
    status[status["CURRENT_STATUS_FLAG"].eq("Y")]
    .drop_duplicates("DRUG_CODE")
    .set_index("DRUG_CODE")[["STATUS", "HISTORY_DATE"]]
)
status["HISTORY_DATE"] = pd.to_datetime(status["HISTORY_DATE"], format="%d-%b-%Y", errors="coerce")
first_marketed = status[status["STATUS"].eq("MARKETED")].groupby("DRUG_CODE")["HISTORY_DATE"].min()
# Approval date = earliest APPROVED or MARKETED entry: APPROVED history starts only in 1979 and
# often records later re-approvals, but a product must be approved before it is marketed
approval_date = status[status["STATUS"].isin(["APPROVED", "MARKETED"])].groupby("DRUG_CODE")["HISTORY_DATE"].min()

products = pd.DataFrame({
    "drug_code": drug["DRUG_CODE"],
    "din": drug["DRUG_IDENTIFICATION_NUMBER"],
    "brand_name": drug["BRAND_NAME"],
    "product_class": drug["CLASS"],
    "number_of_ais": drug["NUMBER_OF_AIS"],
    "extract_set": drug["extract_set"],
}).set_index("drug_code")

products["active_ingredients"] = ingred.groupby("DRUG_CODE")["ingredient_text"].agg(join_unique)
products["manufacturer"] = manufacturer
products["dosage_form"] = tables["form"].groupby("DRUG_CODE")["PHARMACEUTICAL_FORM"].agg(join_unique)
products["route"] = tables["route"].groupby("DRUG_CODE")["ROUTE_OF_ADMINISTRATION"].agg(join_unique)
products["schedule"] = tables["schedule"].groupby("DRUG_CODE")["SCHEDULE"].agg(join_unique)
products["atc_code"] = tables["ther"].groupby("DRUG_CODE")["TC_ATC_NUMBER"].agg(join_unique)
# Yes/No like EMA's "Biosimilar" column; products not in biosimilar.txt are "No"
products["biosimilar"] = products.index.isin(tables["biosimilar"]["DRUG_CODE"]).astype(object)
products["biosimilar"] = products["biosimilar"].map({True: "Yes", False: "No"})
products["current_status"] = current_status["STATUS"]
products["current_status_date"] = pd.to_datetime(current_status["HISTORY_DATE"], format="%d-%b-%Y", errors="coerce")
products["approval_date"] = approval_date
products["first_marketed_date"] = first_marketed
products = products.reset_index()
print(f"Built {len(products)} products")

Saved 58137 products to: /Users/trang.trinh/Library/CloudStorage/OneDrive-Personal/02. MSc - UMA/IE 683 Web Data integration Project/wdi-drug-integration/data/2_translated/dpd_products.csv


## Group products into medicines (brand name + active substances)
`medicine_id` uses the shared key in `medicine_key.py`, e.g. `NEXIUM|ESOMEPRAZOLE`, so Drugs@FDA and EMA can be mapped on the same key later.

In [ ]:
import sys

sys.path.insert(0, str(repo_root / "src" / "data_collection"))
from medicine_key import dpd_base, medicine_id, norm_substance

# Per product: base substance names ("ESOMEPRAZOLE (... MAGNESIUM)" -> "ESOMEPRAZOLE") and strengths
ingred["substance"] = ingred["INGREDIENT"].str.upper().map(dpd_base, na_action="ignore")
ingred["strength"] = strength
substances = ingred.groupby("DRUG_CODE")["substance"].agg(lambda s: list(s.dropna()))
strengths = ingred.groupby("DRUG_CODE")["strength"].agg(lambda s: [x for x in s if x])

products["substances"] = products["drug_code"].map(substances).apply(lambda s: s if isinstance(s, list) else [])
products["strength"] = products["drug_code"].map(strengths).apply(lambda s: s if isinstance(s, list) else [])
products["medicine_id"] = [medicine_id(b, s) for b, s in zip(products["brand_name"], products["substances"])]
products["substance_key"] = products["substances"].apply(lambda s: "; ".join(sorted({norm_substance(x) for x in s})))

# Product level: one row per DIN, medicine_id links each DIN to its medicine
product_columns = ["medicine_id", "drug_code", "din", "brand_name", "product_class", "number_of_ais",
                   "extract_set", "active_ingredients", "manufacturer", "dosage_form", "route", "schedule",
                   "atc_code", "biosimilar", "current_status", "current_status_date", "approval_date", "first_marketed_date"]
products_path = translated_data_dir / "dpd_products.csv"
products[product_columns].to_csv(products_path, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
print(f"Saved {len(products)} products to: {products_path}")

medicines = products.groupby("medicine_id", sort=False).agg(
    brand_name=("brand_name", "first"),
    active_substances=("substance_key", "first"),
    strengths=("strength", lambda s: join_unique(s.explode())),
    licence_holder=("manufacturer", join_unique),
    product_class=("product_class", join_unique),
    atc_code=("atc_code", join_unique),
    biosimilar=("biosimilar", lambda s: "Yes" if s.eq("Yes").any() else "No"),
    schedule=("schedule", join_unique),
    dosage_forms=("dosage_form", join_unique),
    routes=("route", join_unique),
    current_status=("current_status", join_unique),
    approval_date=("approval_date", "min"),
    first_marketed_date=("first_marketed_date", "min"),
    number_of_dins=("din", "nunique"),
).reset_index()

output_path = translated_data_dir / "dpd_medicines.csv"
medicines.to_csv(output_path, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
print(f"Saved {len(medicines)} medicines to: {output_path}")

## Profile

In [ ]:
print(f"Products (DRUG_CODE): {len(products)}  distinct DINs: {products['din'].nunique()}")
print("\nBy extract set:\n" + products["extract_set"].value_counts().to_string())
print("\nBy class:\n" + products["product_class"].value_counts(dropna=False).to_string())
print("\nBy current status:\n" + products["current_status"].value_counts(dropna=False).to_string())

print(f"\nMedicines: {len(medicines)}  (DINs per medicine: mean {medicines['number_of_dins'].mean():.2f}, "
      f"max {medicines['number_of_dins'].max()})")
print(f"Medicines with >1 active substance: {medicines['active_substances'].str.contains(';').sum()}")
print("\nBiosimilar:\n" + medicines["biosimilar"].value_counts().to_string())

profile_cols = medicines.columns.drop("number_of_dins")
profile = pd.DataFrame({
    "missing_percent": (medicines[profile_cols].isna().mean() * 100).round(2),
    # share of medicines with >1 value = conflicts for single-valued attributes
    "multi_value_percent": (medicines[profile_cols].astype("string")
                            .apply(lambda c: c.str.contains("; ", regex=False, na=False)).mean() * 100).round(2),
})
print("\nMedicine table profile:\n" + profile.to_string())
total_cells = medicines[profile_cols].size
missing_cells = int(medicines[profile_cols].isna().sum().sum())
print(f"Total: {missing_cells} of {total_cells} values missing ({missing_cells / total_cells * 100:.2f}%)")

raw_missing = (drug[columns["drug"]].isna().mean() * 100).round(2)
print("\nMissing % in raw drug table (14 columns):\n" + raw_missing.to_string())

# ingredient_names = ingred["INGREDIENT"].dropna().str.upper().drop_duplicates().sort_values()
# human_codes = set(products.loc[products["product_class"].eq("Human"), "drug_code"])
# human_ingredients = ingred.loc[ingred["DRUG_CODE"].isin(human_codes), "INGREDIENT"].dropna().str.upper().nunique()
# print(f"\nDistinct active ingredients: {len(ingredient_names)} (human products only: {human_ingredients})")
# print(f"Products with >1 active ingredient: {(products['number_of_ais'].astype('Int64') > 1).sum()}")

# ingredient_file = translated_data_dir / "dpd_active_ingredients.txt"
# ingredient_file.write_text("\n".join(ingredient_names), encoding="utf-8")
# print(f"Ingredient list saved to: {ingredient_file}")